<a href="https://colab.research.google.com/github/ab23ms233/deep-learning/blob/main/TMDB_reviews_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting up the environment

## Importing libraries

In [2]:
import requests
import pandas as pd
import pprint
import time

# Data Acquisition

## Visualizing data

We send a request to the TMDB API and fetch data. We visualise how the data looks.

In [ ]:
url = "https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=471"
response = requests.get(url)
data = response.json()
pprint.pprint(data)

In [6]:
# data is a dictionary having the following keys
print(data.keys())

dict_keys(['page', 'results', 'total_pages', 'total_results'])


In [29]:
# data['results'] is a list containing the results of all the movies in a page.
data['results'][0]

{'adult': False,
 'backdrop_path': '/gzmKA50LtpHSDYC7lKjUqLtK7Kn.jpg',
 'genre_ids': [28, 80, 53],
 'id': 479040,
 'title': 'Acts of Violence',
 'original_language': 'en',
 'original_title': 'Acts of Violence',
 'overview': 'When his fiancee is kidnapped by human traffickers, Roman and his ex-military brothers set out to track her down and save her before it is too late. Along the way, Roman teams up with Avery, a cop investigating human trafficking and fighting the corrupt bureaucracy that has harmful intentions.',
 'popularity': 2.0057,
 'poster_path': '/x6sBKpLa86CodgTkIK2r8Hnp07m.jpg',
 'release_date': '2018-01-12',
 'softcore': False,
 'video': False,
 'vote_average': 5.571,
 'vote_count': 450}

In [5]:
overview = data['results'][0]['overview']
pprint.pprint(overview)

('When police funding is cut, the Governor announces he must close one of the '
 'academies. To make it fair, the two police academies must compete against '
 "each other to stay in operation. Mauser persuades two officers in Lassard's "
 "academy to better his odds, but things don't quite turn out as expected...")


In [15]:
total_pages = data['total_pages']
total_pages

547

In [ ]:
# Movie data is only present up to page 500. The rest of the pages do not contain anything meaningful
url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page=500"

response = requests.get(url)
data = response.json()
pprint.pprint(data)

## Extracting data

In [30]:
# Function to extract title, overview, and genre_ids of movies from each page
def extract_from_page(num_page, movie_data):
    url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={num_page}"
    response = requests.get(url)
    data = response.json()

    # If page does not contain results
    if 'results' not in data:
        print(f"No 'results' key found on page {num_page}. Skipping this page.")
        return

    # Iterating over the list of results
    for result in data['results']:
        try:
            # Fetching movie data
            overview = result.get('overview')
            title = result.get('title')
            genre_ids = result.get('genre_ids')

            movie_data.append({
                'title': title,
                'overview': overview,
                'genre_ids': genre_ids
            })
        except Exception as e:
            print(f"Error processing movie on page {num_page}: {e}")

In [31]:
movie_data = [] # Initialize an empty list to store dictionaries of movie data

# Fetching movie data from all pages and storing them in a list
for page in range(1, total_pages + 1):
    extract_from_page(page, movie_data)
    time.sleep(0.1)

    if page % 10 == 0:
        print(f"Extracted data till page {page}...")

# After the loop, create the DataFrame from the collected list
df = pd.DataFrame(movie_data)

print()
print(f"Total number of movies extracted: {len(df)}")
display(df.head())

Extracted data till page 10...
Extracted data till page 20...
Extracted data till page 30...
Extracted data till page 40...
Extracted data till page 50...
Extracted data till page 60...
Extracted data till page 70...
Extracted data till page 80...
Extracted data till page 90...
Extracted data till page 100...
Extracted data till page 110...
Extracted data till page 120...
Extracted data till page 130...
Extracted data till page 140...
Extracted data till page 150...
Extracted data till page 160...
Extracted data till page 170...
Extracted data till page 180...
Extracted data till page 190...
Extracted data till page 200...
Extracted data till page 210...
Extracted data till page 220...
Extracted data till page 230...
Extracted data till page 240...
Extracted data till page 250...
Extracted data till page 260...
Extracted data till page 270...
Extracted data till page 280...
Extracted data till page 290...
Extracted data till page 300...
Extracted data till page 310...
Extracted data ti

,title,overview,genre_ids
0,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]"
1,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]"
2,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]"
3,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]"
4,Michael,"The story of Michael Jackson, one of the most ...","[10402, 18]"


## Saving the DataFrame to a JSON file

Use the `.to_json()` method to save the DataFrame to a JSON file. The `orient='records'` parameter ensures that each row is represented as a JSON object, which is a common and easy-to-read format.

In [32]:
df.to_json('movies.json', orient='records', indent=4)
print('DataFrame successfully saved to movies.json')

DataFrame successfully saved to movies.json


## Loading data from a JSON file

To load the data back into a DataFrame, use `pd.read_json()`. If your JSON was saved with `orient='records'`, loading it back in this way will correctly create your DataFrame.

In [4]:
df = pd.read_json('movies.json', orient='records')
print('DataFrame successfully loaded from movies.json')
display(df.head())

DataFrame successfully loaded from movies.json


,title,overview,genre_ids
0,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]"
1,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]"
2,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]"
3,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]"
4,Michael,"The story of Michael Jackson, one of the most ...","[10402, 18]"


# Text Preprocessing

We apply the following preprocessing steps:


1. Lowercasing
2. Removing punctuations
3. Tokenization
4. Stopword removal
5. Stemming



## Lowercasing

In [5]:
df['title'] = df['title'].str.lower()
df['overview'] = df['overview'].str.lower()

display(df['title', 'overview'].head())

'a small woodland creature and a majestic bird, two natural sworn enemies of the valley, magically trade places and set off on an adventure of a lifetime to switch back. their journey soon uncovers a greater threat—one that could endanger not only their species, but the entire valley they call home.'

## Removing punctuations

In [7]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [9]:
punctuations = string.punctuation

# Function to remove punctuation from a given text
def exclude_punc(text):
    for char in text:
        if char in punctuations:
            text = text.replace(char, '')

    return text

df['overview'] = df.apply(exclude_punc, df['overview'])
df['overview'].head()

## Tokenization

In [11]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

df['tokenized_overview'] = df['overview'].apply(word_tokenize)
display(df[['overview', 'tokenized_overview']].head())

,overview,tokenized_overview
0,"a small woodland creature and a majestic bird,...","[a, small, woodland, creature, and, a, majesti..."
1,imprisoned in the 1940s for the double murder ...,"[imprisoned, in, the, 1940s, for, the, double,..."
2,"spanning the years 1945 to 1955, a chronicle o...","[spanning, the, years, 1945, to, 1955, ,, a, c..."
3,science teacher ryland grace wakes up on a spa...,"[science, teacher, ryland, grace, wakes, up, o..."
4,"the story of michael jackson, one of the most ...","[the, story, of, michael, jackson, ,, one, of,..."


## Stopword removal

In [15]:
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [16]:
stop_words = stopwords.words('english')

# Function to remove stop words from tokens
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

df['tokenized_overview'] = df['tokenized_overview'].apply(remove_stopwords)
display(df[['overview', 'tokenized_overview']].head())

,overview,tokenized_overview
0,"a small woodland creature and a majestic bird,...","[small, woodland, creature, majestic, bird, ,,..."
1,imprisoned in the 1940s for the double murder ...,"[imprisoned, 1940s, double, murder, wife, love..."
2,"spanning the years 1945 to 1955, a chronicle o...","[spanning, years, 1945, 1955, ,, chronicle, fi..."
3,science teacher ryland grace wakes up on a spa...,"[science, teacher, ryland, grace, wakes, space..."
4,"the story of michael jackson, one of the most ...","[story, michael, jackson, ,, one, influential,..."


## Stemming

In [19]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

ps.stem('walked')

'walk'

In [20]:
def apply_stemming(tokens):
    return [ps.stem(token) for token in tokens]

df['stemmed_overview'] = df['tokenized_overview'].apply(apply_stemming)
display(df[['overview', 'tokenized_overview', 'stemmed_overview']].head())

,overview,tokenized_overview,stemmed_overview
0,"a small woodland creature and a majestic bird,...","[small, woodland, creature, majestic, bird, ,,...","[small, woodland, creatur, majest, bird, ,, tw..."
1,imprisoned in the 1940s for the double murder ...,"[imprisoned, 1940s, double, murder, wife, love...","[imprison, 1940, doubl, murder, wife, lover, ,..."
2,"spanning the years 1945 to 1955, a chronicle o...","[spanning, years, 1945, 1955, ,, chronicle, fi...","[span, year, 1945, 1955, ,, chronicl, fiction,..."
3,science teacher ryland grace wakes up on a spa...,"[science, teacher, ryland, grace, wakes, space...","[scienc, teacher, ryland, grace, wake, spacesh..."
4,"the story of michael jackson, one of the most ...","[story, michael, jackson, ,, one, influential,...","[stori, michael, jackson, ,, one, influenti, a..."


# Feature Extaction/Text Representation

We use Tf-Idf technique for representing words as vectors

## Concatenating stemmed review tokens

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['stemmed_overview_str'] = df['stemmed_overview'].apply(lambda tokens: ' '.join(tokens))
display(df[['stemmed_overview', 'stemmed_overview_str']].head())

,stemmed_overview,stemmed_overview_str
0,"[small, woodland, creatur, majest, bird, ,, tw...","small woodland creatur majest bird , two natur..."
1,"[imprison, 1940, doubl, murder, wife, lover, ,...","imprison 1940 doubl murder wife lover , upstan..."
2,"[span, year, 1945, 1955, ,, chronicl, fiction,...","span year 1945 1955 , chronicl fiction italian..."
3,"[scienc, teacher, ryland, grace, wake, spacesh...",scienc teacher ryland grace wake spaceship lig...
4,"[stori, michael, jackson, ,, one, influenti, a...","stori michael jackson , one influenti artist w..."


## Applying Tf-Idf

In [28]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['stemmed_overview_str'])
tfidf_array = tfidf_matrix.toarray()

print(tfidf_array)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [29]:
tfidf_array.shape

(10000, 19979)

In [26]:
tfidf.get_feature_names_out()

19979

# Making the recommendation system

## Calculating the similarity matrix between movies

In [30]:
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Shape of similarity matrix: {sim_matrix.shape}")
print(f"First 5x5 segment")
print(sim_matrix[:5, :5])

Shape of similarity matrix: (10000, 10000)
First 5x5 segment
[[1.         0.         0.         0.05222805 0.01508865]
 [0.         1.         0.00530885 0.00989305 0.00982469]
 [0.         0.00530885 1.         0.00778594 0.03644018]
 [0.05222805 0.00989305 0.00778594 1.         0.        ]
 [0.01508865 0.00982469 0.03644018 0.         1.        ]]


## Recommendation function

In [62]:
# Function to calculate similarity between two genres
def genre_similarity(query_genre, recommended_genre):
    q_set = set(query_genre)
    g_set = set(recommended_genre)
    return len(q_set & g_set)/len(q_set | g_set)

# Movie Recommendation Function
def get_recommendations(title, num_recommendations):
    try:
        index = df[df['title'] == title.lower()].index[0]
        genre = df['genre_ids'].iloc[index]
    except Exception:
        print(f"Movie {title} not found in the dataset")
        return

    # Similarity scores for the given movie
    sim_scores = list(enumerate(sim_matrix[index]))
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Movies which are similar to the given one
    top_indices = [i[0] for i in sorted_scores[:num_recommendations+1]]
    top_movies = df['title'].iloc[top_indices].tolist()
    top_movies.remove(title.lower())    # Removing the movie itself from the recommendations

    # Calculating similarity scroe based on overlapping genres
    recommended_genres = df['genre_ids'].iloc[top_indices].tolist()
    scores = [genre_similarity(genre, rgenre) for rgenre in recommended_genres]

    return top_movies, scores


In [63]:
input_movies = ['swapped', 'the godfather', 'project hail mary']

for movie in input_movies:
    rec_movies, scores = get_recommendations(movie, 5)
    print(f"Input movie: {movie}")
    print(f"Recommendations: {rec_movies}")
    print(f"Similarity scores (based on genre overlap): {scores}")
    print()

Input movie: swapped
Recommendations: ['primitive war', 'warriors of the wind', 'we were soldiers', 'night of the comet', 'zambezia']
Similarity scores (based on genre overlap): [1.0, 0.0, 0.75, 0.0, 0.0, 0.6]

Input movie: the godfather
Recommendations: ['the godfather part ii', 'the godfather part iii', 'damaged', 'the alto knights', 'confession of murder']
Similarity scores (based on genre overlap): [1.0, 1.0, 0.6666666666666666, 0.5, 0.6666666666666666, 0.0]

Input movie: project hail mary
Recommendations: ['oxygen', 'the wandering earth', 'sunshine', 'victor frankenstein', "world's greatest dad"]
Similarity scores (based on genre overlap): [1.0, 0.25, 0.25, 0.3333333333333333, 0.25, 0.0]



In [54]:
df['genre_ids'].iloc[3]

[878, 12]